In [22]:
import xgboost as xgb
print(xgb.__version__)

3.1.2


In [1]:
# 12.09 sklearn version + 벡터라이즈 embedding 코드 추가 버전
import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
# current_dir  = os.getcwd()
# project_root = os.path.abspath(os.path.join(current_dir, '..'))
project_root = "c:/big20/git/big20-ML-project2-team3/MercariPriceSuggestions"
# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [10]:
# ============================================================================
# Cell 1: MercariSklearnAnalyzer 완전체 클래스 정의
# ============================================================================
# Mercari Price Suggestion - sklearn 기반 분석 클래스
# 
# 사용법:
#   1. 이 셀 실행 (클래스 정의)
#   2. 다음 셀에서 analyzer = MercariSklearnAnalyzer() 생성
#   3. Pipeline 순차 실행
# ============================================================================

import os
import sys
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime
from typing import List, Dict, Any, Optional
from pathlib import Path  

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor, StackingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from hyperopt import hp
from scipy.sparse import hstack, csr_matrix

warnings.filterwarnings('ignore')

# 프로젝트 루트 추가 (hyperopt_search.py 임포트용)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from utils.hyperopt_search import (
    hyperopt_search,
    train_and_evaluate_regressor,
)

# embeddings 추가
from utils.embeddings import (
    BaseEmbedding,
    TfidfEmbedding,
    Word2VecEmbedding,
    BertEmbedding,
    FastTextEmbedding
)



class MercariSklearnAnalyzer:
    """
    Mercari Price Suggestion - sklearn 기반 완전체 분석 클래스
    
    주요 기능:
    ---------
    1. 데이터 로딩 및 전처리 (TF-IDF + 빈도 인코딩)
    2. Base 모델 학습 (LGBM, XGB, ExtraTrees) + Hyperopt
    3. Stacking 앙상블 (Ridge meta-learner)
    4. 교차 검증 (cross_validate_best)
    5. 피처 중요도 분석 (plot_feature_importance)
    6. 잔차 분석 (residual_analysis)
    7. 캐글 제출 파일 생성
    
    디렉토리 구조:
    -------------
    ../data/          : train.tsv, test.tsv
    ../models/        : 학습된 모델 pickle
    ../results/       : 메트릭 JSON/CSV, 캐시
    ../images/        : 시각화 이미지
    """

    # __init__ start ###########################
    def __init__(
        self,
        random_state: int = 23,
        models_dir: str = "../models",
        results_dir: str = "../results",
        images_dir: str = "../images",
    ):
        """
        초기화 및 기본 경로 설정
        
        Parameters:
        -----------
        random_state : int, default=23
            재현성을 위한 랜덤 시드
        models_dir : str, default="../models"
            모델 저장 경로
        results_dir : str, default="../results"
            결과 저장 경로
        images_dir : str, default="../images"
            시각화 저장 경로
        """
        self.random_state = random_state
        self.models_dir = models_dir
        self.results_dir = results_dir
        self.images_dir = images_dir

        self.train: Optional[pd.DataFrame] = None
        self.test: Optional[pd.DataFrame] = None
        self.vectorizer: Optional[TfidfVectorizer] = None

        # 학습/검증용
        self.X_train = None
        self.X_valid = None
        self.y_train = None
        self.y_valid = None

        # 전체 학습 데이터 (full train)
        self.X_train_full = None
        self.y_train_full = None

        # 캐글 제출용 test 피처
        self.X_test_kaggle = None

        # base_models: { name: { 'model_class', 'params', 'model', 'metrics', ... } }
        self.base_models: Dict[str, Dict[str, Any]] = {}

        # stacking 모델
        self.stack_model: Optional[Any] = None

        # best model
        self.best_model: Optional[Any] = None
        self.best_model_name: Optional[str] = None

        # 모든 모델 메트릭
        self.model_metrics: Dict[str, Dict[str, float]] = {}

        # 최종 best 모델 메트릭
        self.metrics: Dict[str, float] = {}
    # __init__ end ======================================


    # load_data start ###########################
    def load_data(
        self, 
        train_path: str = '../data/train.tsv',  # ✅ 기본값 추가
        test_path: str = '../data/test.tsv',    # ✅ 기본값 추가
        sep: str = "\t"
    ):
        """
        Mercari 데이터 로딩 및 기본 정제
        
        Parameters:
        -----------
        train_path : str, default='../data/train.tsv'
            학습 데이터 경로
        test_path : str, default='../data/test.tsv'
            테스트 데이터 경로
        sep : str, default="\t"
            구분자 (TSV 파일이므로 탭)
        
        Notes:
        ------
        - train: price > 0 필터 + NaN price 제거
        - test: 결측 처리
        - brand_name, category_name, item_description 결측 채우기
        
        Examples:
        ---------
        >>> # 기본 경로 사용
        >>> analyzer.load_data()
        
        >>> # 커스텀 경로
        >>> analyzer.load_data(
        ...     train_path='../raw_data/train.tsv',
        ...     test_path='../raw_data/test.tsv'
        ... )
        """
        self.train = pd.read_csv(train_path, sep=sep)
        self.test = pd.read_csv(test_path, sep=sep)

        # price > 0 필터 + NaN price 제거
        self.train = self.train[self.train["price"] > 0].dropna(subset=["price"])

        # 결측 처리
        for df in [self.train, self.test]:
            df["brand_name"] = df["brand_name"].fillna("Unknown")
            df["category_name"] = df["category_name"].fillna("Unknown")
            df["item_description"] = df["item_description"].fillna("No description")

        print("✅ Data Loaded.")
        print(f"   train: {self.train.shape}, test: {self.test.shape}")
    # load_data end ======================================


    # _split_category_columns start ###########################
    def _split_category_columns(self, df: pd.DataFrame):
        """
        category_name을 'cat1', 'cat2', 'cat3'로 분리
        
        Parameters:
        -----------
        df : pd.DataFrame
            처리할 데이터프레임
        
        Notes:
        ------
        - "Electronics/Computers & Tablets/iPad" → cat1, cat2, cat3
        - 결측값은 "NoCat1", "NoCat2", "NoCat3"로 채움
        """
        cats = df["category_name"].str.split("/", n=2, expand=True)
        df["cat1"] = cats[0].fillna("NoCat1")
        df["cat2"] = cats[1].fillna("NoCat2") if cats.shape[1] > 1 else "NoCat2"
        df["cat3"] = cats[2].fillna("NoCat3") if cats.shape[1] > 2 else "NoCat3"
    # _split_category_columns end ======================================


    # preprocess_all_staged start ###########################
    def preprocess_all_staged(
        self,
        use_cache: bool = True,
        save_cache: bool = True,
        debug: bool = True,
        text_embedding: str = "tfidf",  # "tfidf", "word2vec", "bert"
    ):
        """
        텍스트 + 카테고리/브랜드 + 숫자 피처 전처리 및 벡터화
        
        Parameters:
        -----------
        use_cache : bool, default=True
            캐시 사용 여부 (2회차부터 전처리 생략)
        save_cache : bool, default=True
            캐시 저장 여부
        debug : bool, default=True
            진행 상황 출력 여부
        text_embedding : str, default="tfidf"
            텍스트 임베딩 방식 선택
            - "tfidf": TF-IDF (빠르고 효과적, 기본값)
            - "word2vec": Word2Vec (추후 구현 가능)
            - "bert": BERT (추후 구현 가능)
        
        처리 내용:
        ----------
        1. name + item_description → text_all (임베딩 대상)
        2. category_name 3분할 (cat1, cat2, cat3)
        3. brand_name, cat1~3 빈도 인코딩 (brand_freq, cat1_freq, ...)
        4. 숫자 피처: item_condition_id, shipping, 빈도 피처들
        5. 타깃: log1p(price) 변환
        6. Train/Valid split (80:20)
        7. 캐시: ../results/cache/mercari_preprocessed_{text_embedding}.pkl
        
        Notes:
        ------
        - TF-IDF max_features=30000 (조정 가능)
        - sparse matrix (메모리 효율)
        - 캐시 사용 시 수 초 내 완료
        """
        if self.train is None or self.test is None:
            raise RuntimeError("load_data()를 먼저 호출하세요.")

        # 임베딩 방식 검증
        if text_embedding not in ["tfidf", "word2vec", "bert","fasttext"]:
            raise ValueError(f"지원하지 않는 text_embedding: {text_embedding}")
             

        cache_dir = os.path.join(self.results_dir, "cache")
        os.makedirs(cache_dir, exist_ok=True)
        cache_path = os.path.join(cache_dir, f"mercari_preprocessed_{text_embedding}.pkl")

        if use_cache and os.path.exists(cache_path):
            if debug:
                print(f"📦 캐시 로드: {cache_path}")
            with open(cache_path, "rb") as f:
                (
                    self.X_train,
                    self.X_valid,
                    self.y_train,
                    self.y_valid,
                    self.X_train_full,
                    self.y_train_full,
                    self.X_test_kaggle,
                    self.vectorizer,
                ) = pickle.load(f)
            
            if debug:
                print(f"✅ 캐시 로드 완료 ({text_embedding})")
            return

        # category split
        self._split_category_columns(self.train)
        self._split_category_columns(self.test)

        # text_all 생성
        self.train["text_all"] = (
            self.train["name"].astype(str) + " " +
            self.train["item_description"].astype(str)
        )
        self.test["text_all"] = (
            self.test["name"].astype(str) + " " +
            self.test["item_description"].astype(str)
        )

        # target: log1p(price)
        y_log = np.log1p(self.train["price"].values)

        # 빈도 인코딩용 카운트
        brand_counts = self.train["brand_name"].value_counts()
        cat1_counts = self.train["cat1"].value_counts()
        cat2_counts = self.train["cat2"].value_counts()
        cat3_counts = self.train["cat3"].value_counts()

        # 빈도 인코딩
        self.train["brand_freq"] = self.train["brand_name"].map(brand_counts).fillna(0)
        self.test["brand_freq"] = self.test["brand_name"].map(brand_counts).fillna(0)

        self.train["cat1_freq"] = self.train["cat1"].map(cat1_counts).fillna(0)
        self.test["cat1_freq"] = self.test["cat1"].map(cat1_counts).fillna(0)

        self.train["cat2_freq"] = self.train["cat2"].map(cat2_counts).fillna(0)
        self.test["cat2_freq"] = self.test["cat2"].map(cat2_counts).fillna(0)

        self.train["cat3_freq"] = self.train["cat3"].map(cat3_counts).fillna(0)
        self.test["cat3_freq"] = self.test["cat3"].map(cat3_counts).fillna(0)

        # 숫자 피처
        num_cols = [
            "item_condition_id",
            "shipping",
            "brand_freq",
            "cat1_freq",
            "cat2_freq",
            "cat3_freq",
        ]

        num_train = self.train[num_cols].astype("float32").values
        num_test = self.test[num_cols].astype("float32").values

        # 텍스트 임베딩 (현재는 TF-IDF만 구현)
        if text_embedding == "tfidf":
            self.embedding = TfidfEmbedding(max_features=30000)
        elif text_embedding == "word2vec":
            self.embedding = Word2VecEmbedding(vector_size=300)
        elif text_embedding == "bert":
            self.embedding = BertEmbedding()   # default MiniLM
        elif text_embedding == "fasttext":     # 🔥 추가된 부분
            self.embedding = FastTextEmbedding(vector_size=300)
        else:
            raise ValueError("unknown embedding type")

        # text vectorize
        X_text_train = self.embedding.fit_transform(self.train["text_all"])
        X_text_test = self.embedding.transform(self.test["text_all"])


        # sparse vs dense 분기 - hstack
        if getattr(self.embedding, "is_sparse", False):
            X_full = hstack([X_text_train, csr_matrix(num_train)])
            X_test_kaggle = hstack([X_text_test, csr_matrix(num_test)])
        else:
            X_full = np.hstack([X_text_train, num_train])
            X_test_kaggle = np.hstack([X_text_test, num_test])


        # train/valid split
        X_train, X_valid, y_train, y_valid = train_test_split(
            X_full,
            y_log,
            test_size=0.2,
            random_state=self.random_state,
        )

        self.X_train = X_train
        self.X_valid = X_valid
        self.y_train = y_train
        self.y_valid = y_valid

        self.X_train_full = X_full
        self.y_train_full = y_log
        self.X_test_kaggle = X_test_kaggle

        if save_cache:
            with open(cache_path, "wb") as f:
                pickle.dump(
                    (
                        self.X_train,
                        self.X_valid,
                        self.y_train,
                        self.y_valid,
                        self.X_train_full,
                        self.y_train_full,
                        self.X_test_kaggle,
                        self.vectorizer,
                    ),
                    f,
                )
            if debug:
                print(f"💾 캐시 저장: {cache_path}")

        if debug:
            print(f"✅ 전처리 완료 ({text_embedding})")
            print(f"   - X_train: {X_train.shape}")
            print(f"   - X_valid: {X_valid.shape}")
            print(f"   - X_test_kaggle: {X_test_kaggle.shape}")
    # preprocess_all_staged end ======================================


    # _train_single_model start ###########################
    def _train_single_model(
        self,
        model_name: str,
        model_class,
        search_space: Optional[dict] = None,
        max_evals: int = 50,
        use_hyperopt: bool = True,
        resume_trials: bool = True,  # ✅ 추가: trials 복구 옵션
    ):
        """
        단일 회귀 모델 학습 (내부 메서드)
        
        Parameters:
        -----------
        model_name : str
            모델 식별자 (예: 'lgb', 'xgb', 'et')
        model_class : class
            모델 클래스 (LGBMRegressor, XGBRegressor 등)
        search_space : dict, optional
            hyperopt search space
        max_evals : int, default=50
            hyperopt 최적화 시도 횟수
        use_hyperopt : bool, default=True
            hyperopt 사용 여부
        resume_trials : bool, default=True
            저장된 trials 자동 복구 여부
        
        Notes:
        ------
        1. 기존 trials 파일 존재 시 자동 복구 (resume_trials=True)
        2. hyperopt_search로 하이퍼파라미터 탐색 (선택)
        3. train_and_evaluate_regressor로 최종 학습 + 평가
        4. self.base_models / self.model_metrics 업데이트
        """        
        # ============================================================
        # NEW: 체크 - 이미 저장된 모델이 있으면 학습 skip (가장 최근 모델 사용)
        # ============================================================
        # 파일 패턴 예: LGBMRegressor_reg_20251210_103632.pkl
        model_files = list(Path(self.models_dir).glob(f"{model_class.__name__}_reg_*.pkl"))
        if model_files:
            # 최신 파일 선택
            latest_model_path = max(model_files, key=lambda p: p.stat().st_mtime)
            print(f"📁 저장된 모델 발견 -> 학습 스킵: {latest_model_path.name}")
            
            # 모델 로드
            with open(latest_model_path, 'rb') as f:
                loaded_model = pickle.load(f)

            # 메트릭 JSON도 찾아보자
            # 파일명은 LGBMRegressor_reg_result_***.json 형태
            result_files = list(Path(self.results_dir).glob(f"{model_class.__name__}_reg_result_*.json"))
            metrics = {}
            if result_files:
                latest_result = max(result_files, key=lambda p: p.stat().st_mtime)
                with open(latest_result, 'r', encoding='utf-8') as fr:
                    js = json.load(fr)
                    metrics = js.get("metrics", {})

            # base_models / model_metrics 모두 업데이트
            self.base_models[model_name] = {
                "model_class": model_class,
                "params": {},  # 알 수 없지만 skip 상황에서는 중요하지 않음
                "model": loaded_model,
                "metrics": metrics,
                "model_path": str(latest_model_path),
                "result_path": str(latest_result) if result_files else None,
            }
            self.model_metrics[model_name] = metrics

            # skip training
            return
        
        
        if self.X_train is None or self.y_train is None:
            raise RuntimeError("preprocess_all_staged()를 먼저 호출하세요.")

        print(f"\n🔹 [{model_name}] 회귀 모델 학습 시작")

        # 1) 저장된 trials 복구 시도
        best_params = None
        trials_dir = os.path.join(self.models_dir, 'trials')
        
        if resume_trials and use_hyperopt and search_space is not None:
            # 최신 trials 파일 찾기
            trials_pattern = f"{model_class.__name__}_trials_*.pkl"
            trials_files = list(Path(trials_dir).glob(trials_pattern))
            
            if trials_files:
                # 최신 파일 선택 (파일명의 타임스탬프 기준)
                latest_trials = max(trials_files, key=lambda p: p.stem.split('_')[-2:])
                
                print(f"📦 저장된 trials 발견: {latest_trials.name}")
                try:
                    with open(latest_trials, 'rb') as f:
                        saved_trials = pickle.load(f)
                    
                    # best params 추출
                    if saved_trials.trials:
                        best_idx = np.argmin([t['result']['loss'] for t in saved_trials.trials])
                        best_trial = saved_trials.trials[best_idx]
                        best_score = -best_trial['result']['loss']
                        
                        print(f"✅ Trials 복구 성공!")
                        print(f"   - 시도 횟수: {len(saved_trials.trials)}회")
                        print(f"   - Best 점수: {best_score:.4f}")
                        print(f"   - Hyperopt 생략하고 바로 학습 진행...")
                        
                        # best params 변환 (간단 버전)
                        best_params_raw = best_trial['misc']['vals']
                        best_params = {}
                        
                        integer_params = [
                            'n_estimators', 'max_depth', 'num_leaves', 
                            'min_child_samples', 'min_data_in_leaf'
                        ]
                        
                        for key, value_list in best_params_raw.items():
                            if len(value_list) == 0:
                                continue
                            value = value_list[0]
                            if key in integer_params:
                                best_params[key] = int(value)
                            else:
                                best_params[key] = float(value)
                        
                        best_params['random_state'] = self.random_state
                        
                        print(f"   - 복구된 파라미터: {best_params}")
                        
                except Exception as e:
                    print(f"⚠️ Trials 복구 실패: {e}")
                    print(f"   새로 Hyperopt 실행...")
                    best_params = None

        # 2) hyperopt로 best params 찾기 (복구 실패 시에만)
        if best_params is None and use_hyperopt and search_space is not None:
            result_search = hyperopt_search(
                model_class=model_class,
                search_space=search_space,
                X_train=self.X_train,
                y_train=self.y_train,
                scoring="neg_root_mean_squared_error",
                max_evals=max_evals,
                verbose=True,
                save_trials=True,
                trials_path=trials_dir,
            )
            best_params = result_search["best_params"]
        
        # 3) 기본 파라미터 (Hyperopt 미사용)
        if best_params is None:            
            if model_name == 'et':
                # ExtraTrees는 고정 파라미터 사용
                best_params = {
                    "n_estimators": 200,
                    "max_depth": 15,
                    "max_features": "log2",
                    "random_state": self.random_state,
                    "n_jobs": -1,
                }
                print("👉 ExtraTrees: hyperopt skip, use fixed params:", best_params)
            else:
                best_params = {"random_state": self.random_state}
                print("👉 Default params:", best_params)
                

        # 4) 최종 학습 + 평가 + 저장
        result_train = train_and_evaluate_regressor(
            model_class=model_class,
            params=best_params,
            X_train=self.X_train,
            y_train=self.y_train,
            X_test=self.X_valid,
            y_test=self.y_valid,
            save_model=True,
            save_model_path=self.models_dir,
            save_result_path=self.results_dir,
            verbose=True,
            target_is_log1p=True,
        )

        model = result_train["model"]
        metrics = result_train["metrics"]
        model_path = result_train.get("model_path")
        result_path = result_train.get("result_path")

        print(f"  -> [{model_name}] RMSE: {metrics['rmse']:.4f}, RMSLE: {metrics['rmsle']:.4f}")

        # 5) 내부 저장
        self.base_models[model_name] = {
            "model_class": model_class,
            "params": best_params,
            "model": model,
            "metrics": metrics,
            "model_path": model_path,
            "result_path": result_path,
        }
        self.model_metrics[model_name] = metrics
    # _train_single_model end ======================================


    # train_base_models start ###########################
    def train_base_models(self, use_hyperopt: bool = True, max_evals: int = 50):
        """
        LGBMRegressor, XGBRegressor, ExtraTreesRegressor 3개 base 모델 학습
        
        Parameters:
        -----------
        use_hyperopt : bool, default=True
            hyperopt 사용 여부
        max_evals : int, default=50
            hyperopt 최적화 시도 횟수 (20=빠름, 100+=고성능)
        
        Notes:
        ------
        - LGBM: 빠르고 정확, 대용량 데이터 적합
        - XGB: 강력한 성능, 하이퍼파라미터 민감
        - ExtraTrees: 랜덤성 높음, 다양성 제공
        - 각 모델은 별도로 저장됨 (pickle, JSON)
        """
        # hyperopt search space
        lgb_space = {
            "num_leaves": hp.quniform("num_leaves", 31, 255, 1),
            "learning_rate": hp.loguniform("learning_rate", -5, -1),
            "n_estimators": hp.quniform("n_estimators", 100, 1000, 50),
        }

        xgb_space = {
            "max_depth": hp.quniform("max_depth", 3, 12, 1),
            "learning_rate": hp.loguniform("learning_rate", -5, -1),
            "n_estimators": hp.quniform("n_estimators", 100, 1000, 50),
        }

        et_space = {
            "n_estimators": hp.quniform("n_estimators", 100, 500, 50),
            "max_depth": hp.quniform("max_depth", 5, 20, 1),
        }

        self._train_single_model(
            model_name="lgb",
            model_class=LGBMRegressor,
            search_space=lgb_space,
            max_evals=max_evals,
            use_hyperopt=use_hyperopt,
        )

        self._train_single_model(
            model_name="xgb",
            model_class=XGBRegressor,
            search_space=xgb_space,
            max_evals=max_evals,
            use_hyperopt=use_hyperopt,
        )

        self._train_single_model(
            model_name="et",
            model_class=ExtraTreesRegressor,
            search_space=et_space,
            max_evals=max_evals,
            use_hyperopt=use_hyperopt,
        )
    # train_base_models end ======================================


    # find_best_model start ###########################
    def find_best_model(self):
        """
        base 모델들 중 RMSLE가 가장 작은 모델을 best_model로 선택
        
        Notes:
        ------
        - RMSLE (Root Mean Squared Log Error) 기준
        - Kaggle Mercari 대회의 공식 평가지표
        - 낮을수록 좋음
        """
        if not self.base_models:
            raise RuntimeError("train_base_models()를 먼저 호출하세요.")

        best_name = None
        best_rmsle = float("inf")

        for name, info in self.base_models.items():
            rmsle = info["metrics"]["rmsle"]
            if rmsle < best_rmsle:
                best_rmsle = rmsle
                best_name = name

        self.best_model_name = best_name
        self.best_model = self.base_models[best_name]["model"]
        self.metrics = self.base_models[best_name]["metrics"]

        print(f"\n🏆 Best base model: {best_name}, RMSLE={best_rmsle:.4f}")
    # find_best_model end ======================================


    # stack_models start ###########################
    def stack_models(self):
        """
        base 모델(LGB, XGB, ET)을 이용하여 StackingRegressor 생성
        
        Notes:
        ------
        - Estimators: LGBM, XGB, ExtraTrees
        - Meta-learner: Ridge(alpha=1.0)
        - passthrough=False (base 예측값만 사용)
        - stacking이 기존 best보다 좋으면 자동 교체
        """
        if not self.base_models:
            raise RuntimeError("train_base_models()를 먼저 호출하세요.")

        estimators = [
            (name, info["model"]) for name, info in self.base_models.items()
        ]

        stack = StackingRegressor(
            estimators=estimators,
            final_estimator=Ridge(alpha=1.0),
            passthrough=False,
            n_jobs=-1,
        )

        stack.fit(self.X_train, self.y_train)

        y_pred_log = stack.predict(self.X_valid)
        y_true_log = self.y_valid

        # 원래 price 스케일로 복원 후 메트릭 계산
        y_true = np.expm1(y_true_log)
        y_pred_price = np.maximum(np.expm1(y_pred_log), 0)

        rmse = float(np.sqrt(((y_true - y_pred_price) ** 2).mean()))
        mae = float(np.abs(y_true - y_pred_price).mean())
        r2 = float(np.corrcoef(y_true, y_pred_price)[0, 1] ** 2) if len(y_true) > 1 else 0.0
        rmsle = float(
            np.sqrt(
                np.mean((np.log1p(y_true) - np.log1p(y_pred_price)) ** 2)
            )
        )

        metrics = {
            "rmse": rmse,
            "mae": mae,
            "r2": r2,
            "rmsle": rmsle,
        }

        self.stack_model = stack
        self.base_models["stacking"] = {
            "model_class": None,
            "params": {},
            "model": stack,
            "metrics": metrics,
            "model_path": None,
            "result_path": None,
        }
        self.model_metrics["stacking"] = metrics

        print(f"\n🔷 Stacking RMSE: {rmse:.4f}, RMSLE: {rmsle:.4f}")

        # 기존 best와 비교
        if self.best_model_name is None:
            self.best_model_name = "stacking"
            self.best_model = stack
            self.metrics = metrics
            print("  -> Stacking이 최초 best_model로 설정되었습니다.")
        else:
            best_rmsle = self.model_metrics[self.best_model_name]["rmsle"]
            if rmsle < best_rmsle:
                self.best_model_name = "stacking"
                self.best_model = stack
                self.metrics = metrics
                print("  -> Stacking이 기존 best_model보다 좋아서 교체되었습니다.")
    # stack_models end ======================================


    # evaluate start ###########################
    def evaluate(self) -> Dict[str, float]:
        """
        현재 best_model 기준 메트릭 출력/리턴
        
        Returns:
        --------
        dict : {'rmse', 'mae', 'r2', 'rmsle'}
        """
        if self.best_model_name is None or self.best_model is None:
            raise RuntimeError("best_model이 설정되지 않았습니다.")

        metrics = self.model_metrics.get(self.best_model_name)
        if metrics is None:
            raise RuntimeError(f"model_metrics에 '{self.best_model_name}' 항목이 없습니다.")

        self.metrics = metrics

        print(f"\n📊 Final Evaluation (best_model = {self.best_model_name})")
        for k, v in metrics.items():
            print(f"  - {k}: {v:.4f}")

        return metrics
    # evaluate end ======================================


    # save_best start ###########################
    def save_best(self) -> str:
        """
        best_model을 ../models 아래에 pickle로 저장
        
        Returns:
        --------
        str : 저장된 파일 경로
        """
        if self.best_model is None or self.best_model_name is None:
            raise RuntimeError("저장할 best_model이 없습니다.")

        os.makedirs(self.models_dir, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"best_{self.best_model_name}_{timestamp}.pkl"
        save_path = os.path.join(self.models_dir, filename)

        with open(save_path, "wb") as f:
            pickle.dump(self.best_model, f)

        print(f"💾 Saved best model -> {save_path}")
        return save_path
    # save_best end ======================================


    # save_all_metrics start ###########################
    def save_all_metrics(self) -> Dict[str, str]:
        """
        모든 모델 메트릭을 JSON + CSV로 저장
        
        Returns:
        --------
        dict : {'csv': csv경로, 'results_dir': 디렉토리}
        """
        if not self.model_metrics:
            raise RuntimeError("model_metrics가 비어 있습니다.")

        os.makedirs(self.results_dir, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        json_paths: Dict[str, str] = {}

        # 개별 JSON
        for model_name, metrics in self.model_metrics.items():
            filename = f"metrics_{model_name}_{timestamp}.json"
            save_path = os.path.join(self.results_dir, filename)

            payload = {
                "model_name": model_name,
                "metrics": metrics,
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            }

            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(payload, f, ensure_ascii=False, indent=4)

            json_paths[model_name] = save_path

        # summary CSV
        rows = []
        for model_name, metrics in self.model_metrics.items():
            row = {"model_name": model_name}
            row.update(metrics)
            rows.append(row)

        summary_df = pd.DataFrame(rows)
        csv_filename = f"metrics_summary_{timestamp}.csv"
        csv_path = os.path.join(self.results_dir, csv_filename)
        summary_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

        print("📁 Saved per-model metrics JSON files:")
        for m, p in json_paths.items():
            print(f"  - {m}: {p}")
        print(f"📊 Saved metrics summary CSV -> {csv_path}")

        return {"csv": csv_path, "results_dir": self.results_dir}
    # save_all_metrics end ======================================


    # predict start ###########################
    def predict(self, text_list: List[str]) -> np.ndarray:
        """
        새 텍스트 리스트에 대해 가격 예측
        
        Parameters:
        -----------
        text_list : List[str]
            "name + description" 형식의 텍스트 리스트
        
        Returns:
        --------
        np.ndarray : 예측 가격 (원래 스케일)
        
        Notes:
        ------
        - 수치형 피처는 0으로 채움 (간단한 예측용)
        - 실전에서는 전체 피처 입력 권장
        """
        if self.vectorizer is None:
            raise RuntimeError("vectorizer가 없습니다.")
        if self.best_model is None:
            raise RuntimeError("best_model이 없습니다.")

        X_text = self.vectorizer.transform(text_list)
        num_dummy = np.zeros((len(text_list), 6), dtype="float32")
        
        if self.embedding.is_sparse:
            X_new = hstack([X_text, csr_matrix(num_dummy)])
        else:
            X_new = np.hstack([X_text, num_dummy])
        

        y_pred_log = self.best_model.predict(X_new)
        y_pred_price = np.maximum(np.expm1(y_pred_log), 0)

        return y_pred_price
    # predict end ======================================


    # predict_test_and_save_submission start ###########################
    def predict_test_and_save_submission(self, filename_prefix: str = "submission") -> str:
        """
        캐글 test 데이터 전체 예측 및 submission CSV 저장
        
        Parameters:
        -----------
        filename_prefix : str, default="submission"
            저장 파일명 prefix
        
        Returns:
        --------
        str : 저장된 파일 경로
        
        Notes:
        ------
        - 컬럼: ['test_id', 'price']
        - test_id는 test 데이터의 'test_id' 또는 'id' 컬럼 사용
        """
        if self.best_model is None:
            raise RuntimeError("best_model이 없습니다.")
        if self.X_test_kaggle is None or self.test is None:
            raise RuntimeError("전처리되지 않았거나 test 데이터가 없습니다.")

        y_pred_log = self.best_model.predict(self.X_test_kaggle)
        y_pred_price = np.maximum(np.expm1(y_pred_log), 0)

        if "test_id" in self.test.columns:
            ids = self.test["test_id"].values
        elif "id" in self.test.columns:
            ids = self.test["id"].values
        else:
            ids = np.arange(len(self.test))

        sub_df = pd.DataFrame({"test_id": ids, "price": y_pred_price})

        os.makedirs(self.results_dir, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"{filename_prefix}_{self.best_model_name}_{timestamp}.csv"
        save_path = os.path.join(self.results_dir, filename)

        sub_df.to_csv(save_path, index=False, encoding="utf-8-sig")
        print(f"📄 Saved submission CSV -> {save_path}")

        return save_path
    # predict_test_and_save_submission end ======================================


    # cross_validate_best start ###########################
    def cross_validate_best(self, cv: int = 5, scoring: str = "neg_root_mean_squared_error"):
        """
        Best 모델 K-Fold 교차 검증
        
        Parameters:
        -----------
        cv : int, default=5
            교차 검증 fold 수
        scoring : str, default='neg_root_mean_squared_error'
            평가 지표
        
        Returns:
        --------
        dict : {'cv_scores', 'mean_score', 'std_score', 'cv_rmsle'}
        """
        if self.best_model is None or self.best_model_name is None:
            raise RuntimeError("best_model이 없습니다.")
        
        if self.X_train_full is None or self.y_train_full is None:
            raise RuntimeError("X_train_full이 없습니다.")
        
        print(f"\n{'='*80}")
        print(f"  {self.best_model_name} 모델 {cv}-Fold 교차 검증")
        print(f"{'='*80}")
        
        cv_scores = cross_val_score(
            self.best_model,
            self.X_train_full,
            self.y_train_full,
            cv=cv,
            scoring=scoring,
            n_jobs=-1
        )
        
        if scoring.startswith('neg_'):
            cv_scores = -cv_scores
            metric_name = scoring.replace('neg_', '').upper()
        else:
            metric_name = scoring.upper()
        
        mean_score = cv_scores.mean()
        std_score = cv_scores.std()
        cv_rmsle = mean_score if 'root_mean_squared' in scoring else None
        
        print(f"\n교차 검증 결과 ({metric_name}):")
        print(f"  - Fold별 점수: {cv_scores}")
        print(f"  - 평균: {mean_score:.4f}")
        print(f"  - 표준편차: {std_score:.4f}")
        if cv_rmsle:
            print(f"  - CV RMSLE: {cv_rmsle:.4f} (+/- {std_score:.4f})")
        
        # 시각화
        os.makedirs(self.images_dir, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        fig, ax = plt.subplots(figsize=(10, 6))
        
        folds = [f'Fold {i+1}' for i in range(cv)]
        colors = ['#3498db' if score < mean_score else '#e74c3c' for score in cv_scores]
        bars = ax.bar(folds, cv_scores, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
        
        ax.axhline(mean_score, color='green', linestyle='--', linewidth=2, label=f'Mean: {mean_score:.4f}')
        ax.axhline(mean_score + std_score, color='orange', linestyle=':', linewidth=1.5, label=f'+1 Std')
        ax.axhline(mean_score - std_score, color='orange', linestyle=':', linewidth=1.5, label=f'-1 Std')
        
        ax.set_xlabel('Fold', fontsize=12, fontweight='bold')
        ax.set_ylabel(f'{metric_name} Score', fontsize=12, fontweight='bold')
        ax.set_title(f'{self.best_model_name} - {cv}-Fold CV\nMean: {mean_score:.4f} (+/- {std_score:.4f})', 
                     fontsize=14, fontweight='bold', pad=20)
        ax.legend(loc='upper right', fontsize=10)
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        
        for bar, score in zip(bars, cv_scores):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                    f'{score:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
        
        plt.tight_layout()
        save_path = os.path.join(self.images_dir, f'cv_results_{self.best_model_name}_{timestamp}.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"\n📊 시각화 저장: {save_path}")
        plt.close()
        
        return {
            'cv_scores': cv_scores.tolist(),
            'mean_score': float(mean_score),
            'std_score': float(std_score),
            'cv_rmsle': float(cv_rmsle) if cv_rmsle else None
        }
    # cross_validate_best end ======================================


    # plot_feature_importance start ###########################
    def plot_feature_importance(self, top_n: int = 20, importance_type: str = 'gain'):
        """
        Best 모델 피처 중요도 시각화
        
        Parameters:
        -----------
        top_n : int, default=20
            상위 N개 피처
        importance_type : str, default='gain'
            'gain', 'split', 'weight' 중 선택
        
        Returns:
        --------
        pd.DataFrame : 피처명과 중요도
        """
        if self.best_model is None or self.best_model_name is None:
            raise RuntimeError("best_model이 없습니다.")
        
        if self.vectorizer is None:
            raise RuntimeError("vectorizer가 없습니다.")
        
        print(f"\n{'='*80}")
        print(f"  {self.best_model_name} 모델 피처 중요도 분석")
        print(f"{'='*80}")
        
        tfidf_features = self.vectorizer.get_feature_names_out().tolist()
        numeric_features = [
            'item_condition_id', 'shipping', 'brand_freq',
            'cat1_freq', 'cat2_freq', 'cat3_freq'
        ]
        all_features = tfidf_features + numeric_features
        
        try:
            model = self.best_model
            
            if self.best_model_name == 'stacking':
                print("⚠️ Stacking 모델은 첫 번째 base estimator 사용")
                model = self.best_model.estimators_[0]
            
            if hasattr(model, 'booster_'):
                importances = model.booster_.feature_importance(importance_type=importance_type)
            elif hasattr(model, 'feature_importances_'):
                importances = model.feature_importances_
            else:
                raise AttributeError("피처 중요도를 제공하지 않습니다.")
            
        except Exception as e:
            print(f"❌ 피처 중요도 추출 실패: {e}")
            return None
        
        importance_df = pd.DataFrame({
            'feature': all_features,
            'importance': importances
        }).sort_values('importance', ascending=False).head(top_n)
        
        print(f"\n상위 {top_n}개 중요 피처:")
        print(importance_df.to_string(index=False))
        
        # 시각화
        os.makedirs(self.images_dir, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        fig, ax = plt.subplots(figsize=(12, 8))
        colors = sns.color_palette('viridis', len(importance_df))
        bars = ax.barh(importance_df['feature'], importance_df['importance'], 
                       color=colors, edgecolor='black', linewidth=0.8)
        
        ax.set_xlabel('Importance Score', fontsize=12, fontweight='bold')
        ax.set_ylabel('Feature', fontsize=12, fontweight='bold')
        ax.set_title(f'{self.best_model_name} - Top {top_n} Features\n({importance_type})', 
                     fontsize=14, fontweight='bold', pad=20)
        ax.invert_yaxis()
        ax.grid(axis='x', alpha=0.3, linestyle='--')
        
        for bar, value in zip(bars, importance_df['importance']):
            width = bar.get_width()
            ax.text(width, bar.get_y() + bar.get_height()/2.,
                    f' {value:.1f}', ha='left', va='center', fontsize=9, fontweight='bold')
        
        plt.tight_layout()
        save_path = os.path.join(self.images_dir, f'feature_importance_{self.best_model_name}_{timestamp}.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"\n📊 시각화 저장: {save_path}")
        plt.close()
        
        return importance_df
    # plot_feature_importance end ======================================


    # residual_analysis start ###########################
    def residual_analysis(self, save_samples: bool = True, n_samples: int = 20):
        """
        Best 모델 잔차 분석 및 시각화
        
        Parameters:
        -----------
        save_samples : bool, default=True
            최대/최소 오차 샘플 저장 여부
        n_samples : int, default=20
            저장할 샘플 개수
        
        Returns:
        --------
        dict : {'residuals', 'mean_residual', 'std_residual', ...}
        """
        if self.best_model is None or self.best_model_name is None:
            raise RuntimeError("best_model이 없습니다.")
        
        if self.X_valid is None or self.y_valid is None:
            raise RuntimeError("검증 데이터가 없습니다.")
        
        print(f"\n{'='*80}")
        print(f"  {self.best_model_name} 모델 잔차 분석")
        print(f"{'='*80}")
        
        y_pred_log = self.best_model.predict(self.X_valid)
        y_true_log = self.y_valid
        
        y_true = np.expm1(y_true_log)
        y_pred = np.maximum(np.expm1(y_pred_log), 0)
        
        residuals = y_pred - y_true
        mean_residual = residuals.mean()
        std_residual = residuals.std()
        
        print(f"\n잔차 통계:")
        print(f"  - 평균 잔차: ${mean_residual:.2f}")
        print(f"  - 표준편차: ${std_residual:.2f}")
        print(f"  - 최대 과대예측: ${residuals.max():.2f}")
        print(f"  - 최대 과소예측: ${residuals.min():.2f}")
        
        max_overpredict_idx = residuals.argmax()
        max_underpredict_idx = residuals.argmin()
        
        print(f"\n극단 케이스:")
        print(f"  - 최대 과대예측: 실제=${y_true[max_overpredict_idx]:.2f}, 예측=${y_pred[max_overpredict_idx]:.2f}")
        print(f"  - 최대 과소예측: 실제=${y_true[max_underpredict_idx]:.2f}, 예측=${y_pred[max_underpredict_idx]:.2f}")
        
        # 샘플 저장
        if save_samples:
            os.makedirs(self.results_dir, exist_ok=True)
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            
            sorted_indices = np.argsort(residuals)
            worst_underpredict = sorted_indices[:n_samples]
            worst_overpredict = sorted_indices[-n_samples:]
            sample_indices = np.concatenate([worst_underpredict, worst_overpredict])
            
            samples_df = pd.DataFrame({
                'index': sample_indices,
                'actual_price': y_true[sample_indices],
                'predicted_price': y_pred[sample_indices],
                'residual': residuals[sample_indices],
                'abs_error': np.abs(residuals[sample_indices]),
                'error_type': ['underpredict'] * n_samples + ['overpredict'] * n_samples
            })
            
            csv_path = os.path.join(self.results_dir, f'residual_samples_{self.best_model_name}_{timestamp}.csv')
            samples_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
            print(f"\n💾 샘플 저장: {csv_path}")
        
        # 시각화 (2x2)
        os.makedirs(self.images_dir, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle(f'{self.best_model_name} - Residual Analysis', fontsize=16, fontweight='bold', y=0.995)
        
        # 1. Residual vs Predicted
        ax1 = axes[0, 0]
        ax1.scatter(y_pred, residuals, alpha=0.3, s=10, c='steelblue')
        ax1.axhline(0, color='red', linestyle='--', linewidth=2)
        ax1.set_xlabel('Predicted Price ($)', fontsize=11, fontweight='bold')
        ax1.set_ylabel('Residual ($)', fontsize=11, fontweight='bold')
        ax1.set_title('Residual vs Predicted', fontsize=12, fontweight='bold')
        ax1.grid(alpha=0.3)
        
        # 2. Residual Distribution
        ax2 = axes[0, 1]
        ax2.hist(residuals, bins=50, color='coral', alpha=0.7, edgecolor='black')
        ax2.axvline(mean_residual, color='red', linestyle='--', linewidth=2, label=f'Mean: ${mean_residual:.2f}')
        ax2.axvline(0, color='green', linestyle='-', linewidth=2)
        ax2.set_xlabel('Residual ($)', fontsize=11, fontweight='bold')
        ax2.set_ylabel('Frequency', fontsize=11, fontweight='bold')
        ax2.set_title('Residual Distribution', fontsize=12, fontweight='bold')
        ax2.legend()
        ax2.grid(axis='y', alpha=0.3)
        
        # 3. Actual vs Predicted
        ax3 = axes[1, 0]
        ax3.scatter(y_true, y_pred, alpha=0.3, s=10, c='mediumseagreen')
        max_val = max(y_true.max(), y_pred.max())
        ax3.plot([0, max_val], [0, max_val], 'r--', linewidth=2)
        ax3.set_xlabel('Actual Price ($)', fontsize=11, fontweight='bold')
        ax3.set_ylabel('Predicted Price ($)', fontsize=11, fontweight='bold')
        ax3.set_title('Actual vs Predicted', fontsize=12, fontweight='bold')
        ax3.grid(alpha=0.3)
        
        # 4. Price Range Error
        ax4 = axes[1, 1]
        price_bins = [0, 10, 20, 50, 100, 200, 500, np.inf]
        bin_labels = ['0-10', '10-20', '20-50', '50-100', '100-200', '200-500', '500+']
        price_ranges = pd.cut(y_true, bins=price_bins, labels=bin_labels)
        
        mae_by_range = []
        for label in bin_labels:
            mask = (price_ranges == label)
            mae = np.abs(residuals[mask]).mean() if mask.sum() > 0 else 0
            mae_by_range.append(mae)
        
        colors_range = sns.color_palette('Reds', len(bin_labels))
        bars = ax4.bar(bin_labels, mae_by_range, color=colors_range, alpha=0.8, edgecolor='black')
        
        ax4.set_xlabel('Price Range ($)', fontsize=11, fontweight='bold')
        ax4.set_ylabel('MAE ($)', fontsize=11, fontweight='bold')
        ax4.set_title('MAE by Price Range', fontsize=12, fontweight='bold')
        ax4.grid(axis='y', alpha=0.3)
        
        for bar, mae in zip(bars, mae_by_range):
            if mae > 0:
                ax4.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                        f'${mae:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
        
        plt.tight_layout()
        save_path = os.path.join(self.images_dir, f'residual_analysis_{self.best_model_name}_{timestamp}.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"\n📊 시각화 저장: {save_path}")
        plt.close()
        
        return {
            'residuals': residuals,
            'mean_residual': float(mean_residual),
            'std_residual': float(std_residual),
            'max_overpredict': int(max_overpredict_idx),
            'max_underpredict': int(max_underpredict_idx)
        }
    # residual_analysis end ======================================


print("✅ MercariSklearnAnalyzer 클래스 로드 완료!")
print("   다음 셀에서 analyzer = MercariSklearnAnalyzer()로 시작하세요.")

✅ MercariSklearnAnalyzer 클래스 로드 완료!
   다음 셀에서 analyzer = MercariSklearnAnalyzer()로 시작하세요.


In [11]:
# ============================================================================
# Mercari Price Prediction - Jupyter Notebook 실행 가이드
# ============================================================================
# 각 셀을 순서대로 실행하세요
# ============================================================================

# ============================================================================
# Cell 2: 분석기 초기화
# ============================================================================
analyzer = MercariSklearnAnalyzer(
    random_state=23,
    models_dir="../models",
    results_dir="../results",
    images_dir="../images"
)
print("🚀 Analyzer 초기화 완료!")


🚀 Analyzer 초기화 완료!


In [12]:
# ============================================================================
# Cell 3: 데이터 로딩
# ============================================================================
analyzer.load_data()

✅ Data Loaded.
   train: (1481661, 8), test: (693359, 7)


In [13]:
# ============================================================================
# Cell 4: 전처리 (첫 실행 시 5-10분, 이후 캐시 사용 시 수 초)
# text_embedding = tfidf, word2vec, bert, fasttext
# ============================================================================
analyzer.preprocess_all_staged(
    use_cache=True,      # 캐시 있으면 사용
    save_cache=True,     # 캐시 저장
    debug=True,
    text_embedding="fasttext"
)

print(f"""
전처리 완료:
  - 학습 데이터: {analyzer.X_train.shape}
  - 검증 데이터: {analyzer.X_valid.shape}
  - 테스트 데이터: {analyzer.X_test_kaggle.shape}
""")



📦 캐시 로드: ../results\cache\mercari_preprocessed_fasttext.pkl
✅ 캐시 로드 완료 (fasttext)

전처리 완료:
  - 학습 데이터: (1185328, 306)
  - 검증 데이터: (296333, 306)
  - 테스트 데이터: (1482535, 306)



In [14]:

# ============================================================================
# Cell 5: Base 모델 학습 (3개 모델 + Hyperopt)
# ============================================================================
# 옵션 A: 빠른 테스트 (10-15분, max_evals=20)
analyzer.train_base_models(use_hyperopt=False, max_evals=10)

# 옵션 B: 균형잡힌 성능 (30-40분, max_evals=50)
# analyzer.train_base_models(use_hyperopt=True, max_evals=50)

# 옵션 C: 최고 성능 (2시간+, max_evals=100)
# analyzer.train_base_models(use_hyperopt=True, max_evals=100)

# 옵션 D: Hyperopt 생략 (5분, 낮은 성능)
# analyzer.train_base_models(use_hyperopt=False)



📁 저장된 모델 발견 -> 학습 스킵: LGBMRegressor_reg_20251210_162145.pkl
📁 저장된 모델 발견 -> 학습 스킵: XGBRegressor_reg_20251210_170224.pkl

🔹 [et] 회귀 모델 학습 시작
👉 ExtraTrees: hyperopt skip, use fixed params: {'n_estimators': 200, 'max_depth': 15, 'max_features': 'log2', 'random_state': 23, 'n_jobs': -1}
  ExtraTreesRegressor (regressor) 최종 학습 및 평가
사용 파라미터: {'n_estimators': 200, 'max_depth': 15, 'max_features': 'log2', 'random_state': 23, 'n_jobs': -1}
✓ 학습 완료 (276.09초)

평가 결과 (회귀):
  - rmse: 37.7900
  - mae: 15.0307
  - r2: -0.0005
  - rmsle: 0.6844

✓ 회귀 모델 저장: ../models\ExtraTreesRegressor_reg_20251210_181404.pkl (280.35 MB)
✓ 회귀 결과 저장: ../results\ExtraTreesRegressor_reg_result_20251210_181404.json
  -> [et] RMSE: 37.7900, RMSLE: 0.6844


In [15]:
# ============================================================================
# Cell 6: Best Base 모델 선택
# ============================================================================
analyzer.find_best_model()



🏆 Best base model: xgb, RMSLE=0.4995


In [16]:
# ============================================================================
# Cell 7: Stacking 앙상블
# ============================================================================
analyzer.stack_models()



🔷 Stacking RMSE: 29.6398, RMSLE: 0.4928
  -> Stacking이 기존 best_model보다 좋아서 교체되었습니다.


In [17]:
# ============================================================================
# Cell 8: 최종 평가
# ============================================================================
final_metrics = analyzer.evaluate()

print(f"""
{'='*60}
최종 모델: {analyzer.best_model_name}
{'='*60}
  RMSLE: {final_metrics['rmsle']:.4f}  ← Kaggle 평가지표
  RMSE:  {final_metrics['rmse']:.2f}
  MAE:   {final_metrics['mae']:.2f}
  R²:    {final_metrics['r2']:.4f}
{'='*60}
""")



📊 Final Evaluation (best_model = stacking)
  - rmse: 29.6398
  - mae: 11.0873
  - r2: 0.3989
  - rmsle: 0.4928

최종 모델: stacking
  RMSLE: 0.4928  ← Kaggle 평가지표
  RMSE:  29.64
  MAE:   11.09
  R²:    0.3989



RMSE:  29.64 = 29.6은 나쁘진 않지만, 더 줄일 여지가 많다는 뜻.
  
MAE:   11.09 = 실제 가격과 평균적으로 11달러 차이난다는 의미.  
보통 MAE가 10~12 나오면 준수한 편.  
여기서는 평균 오류가 11달러면 나쁘지 않음.  

R²:    0.3989 = 현재 스태킹 조합이 샘플 편향 or 피처 부족 or 스케일링/파라미터 부조화 등의 영향을 받았을 가능성.  
RMSLE: 0.4928 = 예측값과 실제값의 비율 관계는 꽤 잘 맞는다는 뜻. -> 괜찮은 점수  
⭐ 중간 수준의 모델

MAE & RMSLE는 좋다 → "벗어나지는 않음"

RMSE는 중간 → "오차가 좀 큼"

R²는 낮다 → "스태킹으로 시너지가 충분히 안 남"

즉, 모델이 평균적으로는 괜찮게 예측하지만 변동성이 큰 상품에 대해선 못 맞춘다는 의미.

In [20]:
# ============================================================================
# Cell 9: 모델 및 결과 저장
# ============================================================================
# Best 모델 pickle 저장
best_model_path = analyzer.save_best()

# 모든 모델 메트릭 JSON + CSV 저장
metrics_paths = analyzer.save_all_metrics()

print(f"""
저장 완료:
  📦 Best 모델: {best_model_path}
  📊 메트릭 CSV: {metrics_paths['csv']}
""")



💾 Saved best model -> ../models\best_stacking_20251211_100318.pkl
📁 Saved per-model metrics JSON files:
  - lgb: ../results\metrics_lgb_20251211_100320.json
  - xgb: ../results\metrics_xgb_20251211_100320.json
  - et: ../results\metrics_et_20251211_100320.json
  - stacking: ../results\metrics_stacking_20251211_100320.json
📊 Saved metrics summary CSV -> ../results\metrics_summary_20251211_100320.csv

저장 완료:
  📦 Best 모델: ../models\best_stacking_20251211_100318.pkl
  📊 메트릭 CSV: ../results\metrics_summary_20251211_100320.csv



In [21]:

# ============================================================================
# Cell 15: 전체 결과 요약
# ============================================================================
import pandas as pd

# 메트릭 요약 표 출력
summary = pd.DataFrame([
    {
        '모델': name,
        'RMSLE': f"{metrics['rmsle']:.4f}",
        'RMSE': f"{metrics['rmse']:.2f}",
        'R²': f"{metrics['r2']:.4f}"
    }
    for name, metrics in analyzer.model_metrics.items()
]).sort_values('RMSLE')

print("\n" + "="*70)
print("  전체 모델 성능 비교 (RMSLE 낮을수록 좋음)")
print("="*70)
print(summary.to_string(index=False))
print("="*70)

print(f"""
✅ 분석 완료!

생성된 파일:
  📦 모델: ../models/
  📊 결과: ../results/
  📈 시각화: ../images/
  
다음 단계:
  2. ../images/ 폴더의 시각화 확인
  3. 성능 개선 시도:
     - max_evals 증가 (50 → 100+)
     - TF-IDF max_features 조정 (30000 → 50000)
     - 추가 피처 엔지니어링 (텍스트 길이, 단어 수 등)
""")



  전체 모델 성능 비교 (RMSLE 낮을수록 좋음)
      모델  RMSLE  RMSE      R²
stacking 0.4928 29.64  0.3989
     xgb 0.4995 30.12  0.3644
     lgb 0.5030 30.30  0.3567
      et 0.6844 37.79 -0.0005

✅ 분석 완료!

생성된 파일:
  📦 모델: ../models/
  📊 결과: ../results/
  📈 시각화: ../images/

다음 단계:
  2. ../images/ 폴더의 시각화 확인
  3. 성능 개선 시도:
     - max_evals 증가 (50 → 100+)
     - TF-IDF max_features 조정 (30000 → 50000)
     - 추가 피처 엔지니어링 (텍스트 길이, 단어 수 등)



In [ ]:


# ============================================================================
# [선택] 빠른 테스트 모드 (전체 5분 내 실행)
# ============================================================================
"""
# 새 셀에서 아래 코드 실행 (디버깅용)

analyzer = MercariSklearnAnalyzer(random_state=23)
analyzer.load_data("../data/train.tsv", "../data/test.tsv", sep="\t")
analyzer.preprocess_all_staged(use_cache=True)

# Hyperopt 생략 (기본 파라미터)
analyzer.train_base_models(use_hyperopt=False)
analyzer.find_best_model()
analyzer.stack_models()
analyzer.evaluate()

# 저장
analyzer.save_best()
analyzer.save_all_metrics()
analyzer.predict_test_and_save_submission()

print("✅ 빠른 테스트 완료!")
"""


# ============================================================================
# [선택] 특정 모델만 재학습
# ============================================================================
"""
# LGBM만 다시 학습하고 싶을 때

lgb_space = {
    "num_leaves": hp.quniform("num_leaves", 50, 150, 1),
    "learning_rate": hp.loguniform("learning_rate", -4, -1),
    "n_estimators": hp.quniform("n_estimators", 200, 800, 50),
}

analyzer._train_single_model(
    model_name="lgb_v2",
    model_class=LGBMRegressor,
    search_space=lgb_space,
    max_evals=100,
    use_hyperopt=True
)

# 새로 학습한 모델과 기존 best 비교
print(f"lgb_v2 RMSLE: {analyzer.model_metrics['lgb_v2']['rmsle']:.4f}")
print(f"기존 best RMSLE: {analyzer.metrics['rmsle']:.4f}")
"""